# CH101 free AI 3D autobuild

This notebook creates non-production CH101 review candidates. It uses Stable Fast 3D as the default free Colab provider, with InstantMesh and TripoSR as free fallbacks. Tripo API is optional and never required. It never enables Unity input or approves Gate B.

The adaptive runner checks GPU availability before heavy setup. With a GPU it continues candidate generation; without one it runs the No-GPU validation workstream and records ADAPTIVE_NO_GPU_COMPLETED.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

CHARACTER_CODE = os.environ.get('RE_CAMP_CHARACTER_CODE', 'CH101').upper()
assert CHARACTER_CODE in {'CH101', 'CH102', 'CH103', 'CH104', 'CH105'}
PROVIDER = os.environ.get('RE_CAMP_AI3D_PROVIDER', 'sf3d').lower()
MAX_ATTEMPTS = 3
CANDIDATE_COUNT = int(os.environ.get('RE_CAMP_AI3D_CANDIDATES', '4' if PROVIDER == 'tripo' else '1'))
TOOLS_REPO_URL = 'https://github.com/siri2677/re-camp-blender.git'
TOOLS_REF = os.environ.get('RE_CAMP_BLENDER_TOOLS_REF', 'feature/ch101-free-ai3d-autobuild')
# Set RE_CAMP_BLENDER_TOOLS_COMMIT for a reproducible pin; otherwise resolve the branch tip.
TOOLS_COMMIT = os.environ.get('RE_CAMP_BLENDER_TOOLS_COMMIT', '')
ART_REPO_URL = 'https://github.com/siri2677/re-camp.git'
ART_REF = os.environ.get('RE_CAMP_ART_REF', 'current/art-roster-gate-a-ch102')
ART_COMMIT = 'b6c9b3128358e061eee6184230929413eba84101'
GIT_CLONE_DEPTH = os.environ.get('RE_CAMP_GIT_CLONE_DEPTH', '1')
RUNTIME_NAME = os.environ.get('RE_CAMP_RUNTIME', '').strip().lower()
if not RUNTIME_NAME:
    RUNTIME_NAME = 'kaggle' if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or Path('/kaggle/working').is_dir() else 'colab'
DEFAULT_CONTENT_ROOT = '/kaggle/working' if RUNTIME_NAME == 'kaggle' else '/content'
CONTENT_ROOT = Path(os.environ.get('RE_CAMP_CONTENT_ROOT', DEFAULT_CONTENT_ROOT))

def read_runtime_secret(name):
    value = os.environ.get(name, '').strip()
    if value:
        return value
    if RUNTIME_NAME == 'kaggle':
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name) or ''
        except Exception:
            pass
    if RUNTIME_NAME == 'colab':
        try:
            from google.colab import userdata
            return userdata.get(name) or ''
        except Exception:
            pass
    return ''
TOOLS_DIR = CONTENT_ROOT / 're-camp-blender'
ART_DIR = CONTENT_ROOT / 're-camp'
ROSTER_CONTRACT = TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json'
BASE_CONTRACT = TOOLS_DIR / 'contracts' / 'ch101_ai3d_free_pipeline_v001.json'
OUTPUT_ROOT = CONTENT_ROOT / 're-camp-ai3d' / CHARACTER_CODE
REFERENCE_DIR = OUTPUT_ROOT / 'reference-views'
CANDIDATE_DIR = OUTPUT_ROOT / 'candidates' / PROVIDER
EVALUATION_DIR = OUTPUT_ROOT / 'evaluation'
REVIEW_DIR = OUTPUT_ROOT / 'review'
assert PROVIDER in {'tripo', 'sf3d', 'instantmesh', 'triposr'}
print({'runtime': RUNTIME_NAME, 'provider': PROVIDER, 'maxAttempts': MAX_ATTEMPTS, 'candidateCount': CANDIDATE_COUNT, 'output': str(OUTPUT_ROOT)})


In [ ]:
def run(command, **kwargs):
    print('RUN:', ' '.join(str(part) for part in command), flush=True)
    return subprocess.run([str(part) for part in command], check=True, **kwargs)

run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow'])
# Bootstrap only the small repositories before deciding whether GPU work is allowed.
if not (TOOLS_DIR / '.git').is_dir():
    run(['git', 'clone', '--depth', GIT_CLONE_DEPTH, '--branch', TOOLS_REF, TOOLS_REPO_URL, TOOLS_DIR])
if TOOLS_COMMIT:
    run(['git', '-C', TOOLS_DIR, 'fetch', 'origin', TOOLS_COMMIT])
    run(['git', '-C', TOOLS_DIR, 'checkout', '--detach', TOOLS_COMMIT])
else:
    run(['git', '-C', TOOLS_DIR, 'fetch', 'origin', TOOLS_REF])
    run(['git', '-C', TOOLS_DIR, 'checkout', '--detach', f'origin/{TOOLS_REF}'])
    TOOLS_COMMIT = subprocess.check_output(['git', '-C', TOOLS_DIR, 'rev-parse', 'HEAD'], text=True).strip()
if not (ART_DIR / '.git').is_dir():
    run(['git', 'clone', '--depth', GIT_CLONE_DEPTH, '--branch', ART_REF, ART_REPO_URL, ART_DIR])
run(['git', '-C', ART_DIR, 'fetch', '--depth', GIT_CLONE_DEPTH, 'origin', ART_COMMIT])
run(['git', '-C', ART_DIR, 'checkout', '--detach', ART_COMMIT])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ADAPTIVE_REPORT_PATH = OUTPUT_ROOT / 'adaptive-workstream-report.json'
run([sys.executable, TOOLS_DIR / 'scripts' / 'run_adaptive_workstream.py', '--provider', PROVIDER, '--character', CHARACTER_CODE, '--art-root', ART_DIR, '--output', ADAPTIVE_REPORT_PATH])
adaptive_report = json.loads(ADAPTIVE_REPORT_PATH.read_text(encoding='utf-8'))
GPU_WORK_ENABLED = adaptive_report['selectedWorkstream'] in {'GPU', 'NON_GPU_PROVIDER'}
print({'adaptiveStatus': adaptive_report['status'], 'selectedWorkstream': adaptive_report['selectedWorkstream'], 'gpuWorkEnabled': GPU_WORK_ENABLED, 'unityInputAllowed': False})
if not GPU_WORK_ENABLED:
    raise RuntimeError('ADAPTIVE_NO_GPU_COMPLETED: maintenance work finished; rerun this Notebook when a GPU is available')

if shutil.which('blender') is None or shutil.which('xvfb-run') is None:
    run(['apt-get', 'update', '-qq'])
    run(['apt-get', 'install', '-y', '-qq', 'blender', 'xvfb'])
if not (TOOLS_DIR / '.git').is_dir():
    run(['git', 'clone', '--depth', GIT_CLONE_DEPTH, '--branch', TOOLS_REF, TOOLS_REPO_URL, TOOLS_DIR])
if TOOLS_COMMIT:
    run(['git', '-C', TOOLS_DIR, 'fetch', 'origin', TOOLS_COMMIT])
    run(['git', '-C', TOOLS_DIR, 'checkout', '--detach', TOOLS_COMMIT])
else:
    run(['git', '-C', TOOLS_DIR, 'fetch', 'origin', TOOLS_REF])
    run(['git', '-C', TOOLS_DIR, 'checkout', '--detach', f'origin/{TOOLS_REF}'])
    TOOLS_COMMIT = subprocess.check_output(['git', '-C', TOOLS_DIR, 'rev-parse', 'HEAD'], text=True).strip()
if not (ART_DIR / '.git').is_dir():
    run(['git', 'clone', '--depth', GIT_CLONE_DEPTH, '--branch', ART_REF, ART_REPO_URL, ART_DIR])
run(['git', '-C', ART_DIR, 'fetch', '--depth', GIT_CLONE_DEPTH, 'origin', ART_COMMIT])
run(['git', '-C', ART_DIR, 'checkout', '--detach', ART_COMMIT])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('blender:', shutil.which('blender'))
# Keep Blender's isolated site path available without a second package-manager bootstrap.
BLENDER_PYTHON_SITE = CONTENT_ROOT / 'blender-python-site'
BLENDER_PYTHON_SITE.mkdir(parents=True, exist_ok=True)
# The repository's Blender scripts use bpy/mathutils only; keep the
# Kaggle setup free of a second Python package-manager/network bootstrap.
BLENDER_ENVIRONMENT = os.environ.copy()
BLENDER_ENVIRONMENT['PYTHONPATH'] = str(BLENDER_PYTHON_SITE)


In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
prepare_script = TOOLS_DIR / 'scripts' / 'ai3d' / 'prepare_reference_views.py'
run([
    sys.executable, prepare_script,
    '--art-root', ART_DIR,
    '--output-dir', REFERENCE_DIR,
    '--contract', ROSTER_CONTRACT,
    '--character', CHARACTER_CODE,
])
REFERENCE_MANIFEST = REFERENCE_DIR / 'reference-views-manifest.json'
reference_manifest = json.loads(REFERENCE_MANIFEST.read_text(encoding='utf-8'))
assert reference_manifest['artCommit'] == ART_COMMIT
assert reference_manifest['unityInputAllowed'] is False
roster_payload = json.loads(ROSTER_CONTRACT.read_text(encoding='utf-8'))
character_entry = next(entry for entry in roster_payload['characters'] if entry['character'] == CHARACTER_CODE)
generation_strategy = character_entry.get('generationStrategy', {})
print({'generationProfile': generation_strategy.get('profile', 'DEFAULT'), 'auxiliaryReferenceCount': len(reference_manifest.get('auxiliaryReferences', [])), 'unityInputAllowed': False})
print(json.dumps(reference_manifest, indent=2, ensure_ascii=False))


In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
CANDIDATE_MANIFESTS = []
attempt_summaries = []
provider_environment = os.environ.copy()
if PROVIDER == 'tripo':
    api_key = read_runtime_secret('TRIPO_API_KEY')
    command = [
        sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'tripo_api.py',
        '--reference-manifest', REFERENCE_MANIFEST,
        '--output-dir', CANDIDATE_DIR,
        '--contract', ROSTER_CONTRACT,
        '--character', CHARACTER_CODE,
        '--candidate-count', str(CANDIDATE_COUNT),
    ]
    if api_key:
        provider_environment['TRIPO_API_KEY'] = api_key
        command.append('--execute')
    else:
        print('TRIPO_API_KEY is absent: creating a zero-credit dry-run plan only.')
    run(command, env=provider_environment)
    candidate_manifest_path = CANDIDATE_DIR / 'candidate-manifest.json'
    if candidate_manifest_path.is_file():
        CANDIDATE_MANIFESTS.append(candidate_manifest_path)
else:
    PREFLIGHT_REPORT = OUTPUT_ROOT / 'runtime-preflight.json'
    run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'colab_runtime_preflight.py', '--provider', PROVIDER, '--output', PREFLIGHT_REPORT])
    runtime_preflight = json.loads(PREFLIGHT_REPORT.read_text(encoding='utf-8'))
    torch_info = runtime_preflight.get('torch', {})
    device_capability = torch_info.get('deviceCapability') or {}
    GPU_CUDA_ARCH_LIST = os.environ.get('RE_CAMP_CUDA_ARCH_LIST', device_capability.get('label', '7.5'))
    TORCH_KERNEL_SUPPORTS_DEVICE = bool(torch_info.get('torchKernelSupportsDevice', False))
    print({'deviceName': torch_info.get('deviceName', ''), 'deviceCapability': device_capability.get('label', ''), 'torchKernelSupportsDevice': TORCH_KERNEL_SUPPORTS_DEVICE, 'cudaArchList': GPU_CUDA_ARCH_LIST})
    if device_capability.get('major') == 6 and not TORCH_KERNEL_SUPPORTS_DEVICE:
        legacy_cuda_index = os.environ.get('RE_CAMP_LEGACY_TORCH_INDEX', 'https://download.pytorch.org/whl/cu118')
        print('P100 detected: attempting a Python 3.12-compatible PyTorch CUDA 11.8 build before provider setup.')
        try:
            run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', 'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1', '--index-url', legacy_cuda_index])
        except subprocess.CalledProcessError as error:
            print(f'Legacy PyTorch compatibility install failed; provider work remains blocked: {error}')
        run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'colab_runtime_preflight.py', '--provider', PROVIDER, '--output', PREFLIGHT_REPORT])
        runtime_preflight = json.loads(PREFLIGHT_REPORT.read_text(encoding='utf-8'))
        torch_info = runtime_preflight.get('torch', {})
        device_capability = torch_info.get('deviceCapability') or device_capability
        GPU_CUDA_ARCH_LIST = os.environ.get('RE_CAMP_CUDA_ARCH_LIST', device_capability.get('label', GPU_CUDA_ARCH_LIST))
        TORCH_KERNEL_SUPPORTS_DEVICE = bool(torch_info.get('torchKernelSupportsDevice', False))
        print({'compatibilityTorch': torch_info.get('version', ''), 'torchKernelSupportsDevice': TORCH_KERNEL_SUPPORTS_DEVICE, 'cudaArchList': GPU_CUDA_ARCH_LIST})
    LEGACY_PASCAL_GPU = device_capability.get('major') == 6
    def ensure_legacy_torch():
        if not LEGACY_PASCAL_GPU:
            return
        probe = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__); print(torch.cuda.get_arch_list() if torch.cuda.is_available() else [])'], capture_output=True, text=True, check=False)
        if '2.7.1+cu118' not in probe.stdout or 'sm_60' not in probe.stdout:
            run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', 'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1', '--index-url', os.environ.get('RE_CAMP_LEGACY_TORCH_INDEX', 'https://download.pytorch.org/whl/cu118')])
        print('Legacy torch probe:', subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__); print(torch.cuda.get_arch_list() if torch.cuda.is_available() else [])'], capture_output=True, text=True, check=False).stdout.strip())
    contract = json.loads(BASE_CONTRACT.read_text(encoding='utf-8'))
    hf_token = read_runtime_secret('HF_TOKEN')
    def record_provider_skip(provider_name, output_dir, status, reason):
        output_dir.mkdir(parents=True, exist_ok=True)
        (output_dir / 'provider-skip.json').write_text(json.dumps({'provider': provider_name, 'status': status, 'reason': reason, 'referenceManifest': str(REFERENCE_MANIFEST.resolve()), 'artCommit': ART_COMMIT, 'toolsCommit': TOOLS_COMMIT, 'unityInputAllowed': False, 'productionPromotionAllowed': False}, indent=2) + '\n', encoding='utf-8')
    for attempt in range(1, MAX_ATTEMPTS + 1):
        provider_attempts = [PROVIDER, 'triposr', 'instantmesh'] if PROVIDER == 'sf3d' and LEGACY_PASCAL_GPU else ([PROVIDER, 'instantmesh', 'triposr'] if PROVIDER == 'sf3d' else [PROVIDER])
        attempt_succeeded = False
        for attempt_provider in provider_attempts:
            provider_environment = os.environ.copy()
            if attempt_provider == 'sf3d':
                provider_key = 'stableFast3D'
            elif attempt_provider == 'instantmesh':
                provider_key = 'instantMesh'
            else:
                provider_key = 'tripoSR'
            provider_config = contract['providers'][provider_key]
            provider_repo = CONTENT_ROOT / f"provider-{attempt_provider}"
            attempt_output_dir = OUTPUT_ROOT / 'attempts' / f'{attempt:02d}' / 'candidates' / attempt_provider
            attempt_output_dir.mkdir(parents=True, exist_ok=True)
            if attempt_provider == 'sf3d' and not hf_token:
                record_provider_skip(attempt_provider, attempt_output_dir, 'SKIPPED_MISSING_HF_TOKEN', 'HF_GATED_MODEL_AUTH_REQUIRED')
                print('Skipping SF3D: HF_TOKEN is absent and the pinned model is gated; fallback remains enabled.')
                continue
            if attempt_provider == 'triposr' and not TORCH_KERNEL_SUPPORTS_DEVICE:
                record_provider_skip(attempt_provider, attempt_output_dir, 'SKIPPED_TORCH_DEVICE_INCOMPATIBLE', 'CUDA_KERNEL_NOT_COMPILED_FOR_DEVICE')
                print('Skipping TripoSR: the installed PyTorch wheel has no kernel for the visible GPU; use a compatible runtime before retrying.')
                continue
            if attempt_provider == 'instantmesh' and not TORCH_KERNEL_SUPPORTS_DEVICE:
                record_provider_skip(attempt_provider, attempt_output_dir, 'SKIPPED_TORCH_DEVICE_INCOMPATIBLE', 'CUDA_KERNEL_NOT_COMPILED_FOR_DEVICE')
                print('Skipping InstantMesh: the installed PyTorch wheel has no kernel for the visible GPU; use a compatible runtime before retrying.')
                continue
            if attempt_provider == 'instantmesh' and LEGACY_PASCAL_GPU:
                record_provider_skip(attempt_provider, attempt_output_dir, 'SKIPPED_NVDIFFRAST_TOOLKIT_INCOMPATIBLE', 'CUDA_TOOLKIT_MISMATCH_FOR_LEGACY_PASCAL_GPU')
                print('Skipping InstantMesh on Pascal GPU: nvdiffrast requires a CUDA toolkit compatible with the legacy torch runtime; TripoSR remains the fallback.')
                continue
            existing_manifest = attempt_output_dir / 'candidate-manifest.json'
            if os.environ.get('RE_CAMP_REUSE_CANDIDATES', '1') == '1' and existing_manifest.is_file():
                try:
                    existing = json.loads(existing_manifest.read_text(encoding='utf-8'))
                    existing_candidates = existing.get('candidates', [])
                    reusable = (
                        existing.get('unityInputAllowed') is False
                        and existing.get('provider') == provider_key
                        and existing.get('referenceManifest') == str(REFERENCE_MANIFEST.resolve())
                        and existing_candidates
                        and all(entry.get('status') == 'DOWNLOADED' and Path(entry.get('modelPath', '')).is_file() for entry in existing_candidates)
                    )
                    if reusable:
                        print(f'Reusing existing {attempt_provider} candidate manifest: {existing_manifest}')
                        CANDIDATE_MANIFESTS.append(existing_manifest)
                        attempt_summaries.append({'attempt': attempt, 'provider': attempt_provider, 'referenceView': 'reused', 'foregroundRatio': None, 'manifest': str(existing_manifest), 'status': 'REUSED'})
                        attempt_succeeded = True
                        break
                except (OSError, json.JSONDecodeError, TypeError):
                    print(f'Existing manifest is not reusable; continuing: {existing_manifest}')
            if not (provider_repo / '.git').is_dir():
                run(['git', 'clone', provider_config['repository'], provider_repo])
            run(['git', '-C', provider_repo, 'fetch', 'origin', provider_config['commit']])
            run(['git', '-C', provider_repo, 'checkout', '--detach', provider_config['commit']])
            if attempt_provider == 'instantmesh':
                try:
                    # nvdiffrast builds a PyTorch/CUDA extension. Colab's packaged
                    # CUDA layout requires a non-isolated build and Ninja.
                    provider_environment['CUDA_HOME'] = os.environ.get('CUDA_HOME', '/usr/local/cuda')
                    provider_environment['CUDA_PATH'] = provider_environment['CUDA_HOME']
                    provider_environment.setdefault('TORCH_CUDA_ARCH_LIST', GPU_CUDA_ARCH_LIST)
                    run([sys.executable, '-m', 'pip', 'install', '-q', 'setuptools', 'wheel', 'ninja'], env=provider_environment)
                    run([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-lightning==2.1.2', 'gradio==3.41.2', 'huggingface-hub', 'einops', 'omegaconf', 'torchmetrics', 'webdataset', 'accelerate', 'tensorboard', 'PyMCubes', 'trimesh>=4.4.0', 'rembg', 'transformers==4.34.1', 'diffusers==0.20.2', 'bitsandbytes', 'imageio[ffmpeg]', 'xatlas', 'plyfile'], env=provider_environment)
                    # InstantMesh imports this helper; Colab images may retain an older hub package.
                    run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'huggingface-hub>=0.26.0,<1.0'], env=provider_environment)
                    instantmesh_compat_dir = CONTENT_ROOT / 'instantmesh-hf-compat'
                    instantmesh_compat_dir.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(TOOLS_DIR / 'scripts' / 'ai3d' / 'instantmesh_hf_compat_sitecustomize.py', instantmesh_compat_dir / 'sitecustomize.py')
                    provider_environment['PYTHONPATH'] = os.pathsep.join([str(instantmesh_compat_dir), provider_environment.get('PYTHONPATH', '')])
                    run([sys.executable, '-c', 'from huggingface_hub import split_torch_state_dict_into_shards; from diffusers import DiffusionPipeline'], env=provider_environment)
                    try:
                        import nvdiffrast.torch  # noqa: F401
                        run([sys.executable, '-c', 'import torch; import nvdiffrast.torch as dr; dr.RasterizeCudaContext(device=torch.device(\"cuda:0\"))'], env=provider_environment)
                    except (ImportError, subprocess.CalledProcessError):
                        run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'nvdiffrast'], env=provider_environment)
                        run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', 'git+https://github.com/NVlabs/nvdiffrast/'], env=provider_environment)
                except subprocess.CalledProcessError as error:
                    print(f'InstantMesh setup failed; continuing to next fallback: {error}')
                    continue
            elif attempt_provider == 'sf3d':
                # Colab compatibility: replace the broken gpytoolbox source pin.
                sf3d_requirements = CONTENT_ROOT / 'sf3d-colab-requirements.txt'
                sf3d_lines = [line for line in (provider_repo / 'requirements.txt').read_text(encoding='utf-8').splitlines() if not line.startswith('gpytoolbox==')]
                sf3d_requirements.write_text('\n'.join(sf3d_lines) + '\n', encoding='utf-8')
                run([sys.executable, '-m', 'pip', 'install', '-q', 'cupy-cuda12x==13.6.0'])
                run([sys.executable, '-m', 'pip', 'install', '-q', 'gpytoolbox==0.3.3'])
                run([sys.executable, '-m', 'pip', 'install', '-q', '-r', sf3d_requirements], cwd=provider_repo)
            else:
                run([sys.executable, '-m', 'pip', 'install', '-q', 'omegaconf==2.3.0', 'einops==0.7.0', 'transformers==4.35.0', 'trimesh>=4.4.0', 'rembg', 'imageio[ffmpeg]', 'gradio', 'xatlas==0.0.9', 'moderngl==5.10.0'])
                run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--no-deps', 'huggingface-hub==0.25.2'])
                import importlib.util
                if importlib.util.find_spec('onnxruntime') is None:
                    run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime-gpu'])
                try:
                    import torchmcubes  # noqa: F401
                except ImportError:
                    run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/tatsy/torchmcubes.git'])
                if LEGACY_PASCAL_GPU:
                    ensure_legacy_torch()
                    run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-build-core', 'pybind11', 'cmake', 'ninja'])
                    run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchmcubes'])
                    try:
                        run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', 'git+https://github.com/tatsy/torchmcubes.git'])
                    except subprocess.CalledProcessError as error:
                        record_provider_skip(attempt_provider, attempt_output_dir, 'FAILED_TORCHMCUBES_BUILD', 'TORCHMCUBES_BUILD_FAILED')
                        print(f'TripoSR torchmcubes build failed; continuing to the next attempt: {error}')
                        continue
            if hf_token:
                provider_environment['HF_TOKEN'] = hf_token
            if attempt_provider == 'sf3d' and not provider_environment.get('HF_TOKEN'):
                print('SF3D model access may be gated: set the runtime secret HF_TOKEN and accept the Hugging Face model access terms; fallback remains enabled.')
            try:
                provider_command = [sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'run_open_source_provider.py', '--provider', attempt_provider, '--provider-repo', provider_repo, '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', attempt_output_dir, '--contract', ROSTER_CONTRACT, '--character', CHARACTER_CODE, '--execute']
                reference_view = 'front'
                foreground_ratio = None
                if attempt_provider in {'instantmesh', 'triposr'}:
                    reference_views = generation_strategy.get('singleViewReferenceSequence', provider_config.get('referenceViews', ['front', 'right', 'back']))
                    reference_view = reference_views[(attempt - 1) % len(reference_views)]
                    provider_command.extend(['--reference-view', reference_view])
                    if attempt_provider == 'triposr':
                        foreground_ratios = generation_strategy.get('foregroundRatios', provider_config.get('foregroundRatios', [0.85]))
                        foreground_ratio = foreground_ratios[(attempt - 1) % len(foreground_ratios)]
                        provider_command.extend(['--foreground-ratio', str(foreground_ratio)])
                run(provider_command, env=provider_environment)
                manifest = attempt_output_dir / 'candidate-manifest.json'
                if not manifest.is_file():
                    raise RuntimeError(f'provider produced no candidate manifest: {manifest}')
                CANDIDATE_MANIFESTS.append(manifest)
                # Keep later attempts on the provider that actually succeeded.
                PROVIDER = attempt_provider
                attempt_summaries.append({'attempt': attempt, 'provider': attempt_provider, 'referenceView': reference_view, 'foregroundRatio': foreground_ratio, 'generationProfile': generation_strategy.get('profile', 'DEFAULT'), 'manifest': str(manifest), 'status': 'GENERATED'})
                attempt_succeeded = True
                break
            except (subprocess.CalledProcessError, RuntimeError) as error:
                print(f'Attempt {attempt} provider {attempt_provider} failed: {error}')
        if not attempt_succeeded:
            attempt_summaries.append({'attempt': attempt, 'status': 'FAILED'})
    if not CANDIDATE_MANIFESTS:
        raise RuntimeError('all AI candidate attempts failed')
print(json.dumps({'candidateManifests': [str(path) for path in CANDIDATE_MANIFESTS], 'attempts': attempt_summaries, 'unityInputAllowed': False}, indent=2, ensure_ascii=False))


In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
score_reports = []
refinement_reports = []
REFINEMENT_STATUS = 'REFINED_REVIEW_CANDIDATE'
face_driver_status = 'BLOCKED_NO_RELIABLE_FREE_FACE_LANDMARK_TRANSFER'
socket_review_status = 'AUTO_ESTIMATED_NOT_APPROVED'
candidate_manifests = [Path(path) for path in CANDIDATE_MANIFESTS if Path(path).is_file()]
if candidate_manifests:
    launcher = ['xvfb-run', '-a'] if shutil.which('xvfb-run') else []
    for manifest_path in candidate_manifests:
        candidate_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        assert candidate_manifest['unityInputAllowed'] is False
        attempt_labels = [parent.name for parent in manifest_path.parents if parent.name.isdigit()]
        attempt_label = attempt_labels[0] if attempt_labels else 'attempt_00'
        attempt_number = int(attempt_label) if attempt_label.isdigit() else 0
        candidates = [entry for entry in candidate_manifest.get('candidates', []) if entry.get('status') == 'DOWNLOADED']
        for entry in candidates:
            source_candidate_id = entry['candidateId']
            candidate_id = f'{attempt_label}-{source_candidate_id}'
            candidate_output = EVALUATION_DIR / candidate_id
            candidate_output.mkdir(parents=True, exist_ok=True)
            refined_glb = candidate_output / f'{candidate_id}_refined.glb'
            refined_blend = candidate_output / f'{candidate_id}_refined_NOT_PRODUCTION.blend'
            refinement_report = candidate_output / 'refinement-report.json'
            try:
                run(launcher + [
                    'blender', '-b',
                    '--python', TOOLS_DIR / 'scripts' / 'blender' / 'refine_ai3d_candidate.py',
                    '--',
                    '--candidate', entry['modelPath'],
                    '--character', CHARACTER_CODE,
                    '--output-glb', refined_glb,
                    '--output-blend', refined_blend,
                    '--report', refinement_report,
                    '--provider', candidate_manifest.get('provider', 'unknown'),
                    '--attempt', str(attempt_number),
                    '--parent-sha256', entry.get('sha256', ''),
                    '--material-mode', 'preserve',
                ], env=BLENDER_ENVIRONMENT)
                refinement_reports.append(refinement_report)
                evaluation_report = candidate_output / 'evaluation-report.json'
                normalized_blend = candidate_output / f'{candidate_id}_normalized_NOT_PRODUCTION.blend'
                run(launcher + [
                    'blender', '-b',
                    '--python', TOOLS_DIR / 'scripts' / 'blender' / 'evaluate_ai3d_candidate.py',
                    '--',
                    '--candidate', refined_glb,
                    '--character', CHARACTER_CODE,
                    '--candidate-id', candidate_id,
                    '--output-dir', candidate_output,
                    '--report', evaluation_report,
                    '--normalized-blend', normalized_blend,
                    '--integrity-blend', refined_blend,
                ], env=BLENDER_ENVIRONMENT)
                score_report = candidate_output / 'candidate-score.json'
                run([
                    sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'score_candidate_renders.py',
                    '--reference-manifest', REFERENCE_MANIFEST,
                    '--evaluation-report', evaluation_report,
                    '--output', score_report,
                    '--contract', ROSTER_CONTRACT,
                    '--character', CHARACTER_CODE,
                ])
                score_data = json.loads(score_report.read_text(encoding='utf-8'))
                if score_data.get('orientationValidation', {}).get('correctionRequired'):
                    print(f'Upside-down candidate detected; applying vertical polarity correction: {candidate_id}')
                    run(launcher + [
                        'blender', '-b',
                        '--python', TOOLS_DIR / 'scripts' / 'blender' / 'refine_ai3d_candidate.py',
                        '--',
                        '--candidate', entry['modelPath'],
                        '--character', CHARACTER_CODE,
                        '--output-glb', refined_glb,
                        '--output-blend', refined_blend,
                        '--report', refinement_report,
                        '--provider', candidate_manifest.get('provider', 'unknown'),
                        '--attempt', str(attempt_number),
                        '--parent-sha256', entry.get('sha256', ''),
                        '--material-mode', 'preserve',
                        '--invert-up-axis',
                    ], env=BLENDER_ENVIRONMENT)
                    run(launcher + [
                        'blender', '-b',
                        '--python', TOOLS_DIR / 'scripts' / 'blender' / 'evaluate_ai3d_candidate.py',
                        '--',
                        '--candidate', refined_glb,
                        '--character', CHARACTER_CODE,
                        '--candidate-id', candidate_id,
                        '--output-dir', candidate_output,
                        '--report', evaluation_report,
                        '--normalized-blend', normalized_blend,
                        '--integrity-blend', refined_blend,
                    ], env=BLENDER_ENVIRONMENT)
                    run([
                        sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'score_candidate_renders.py',
                        '--reference-manifest', REFERENCE_MANIFEST,
                        '--evaluation-report', evaluation_report,
                        '--output', score_report,
                        '--contract', ROSTER_CONTRACT,
                        '--character', CHARACTER_CODE,
                    ])
                    score_data = json.loads(score_report.read_text(encoding='utf-8'))
                    if score_data.get('orientationValidation', {}).get('correctionRequired'):
                        raise RuntimeError(f'VERTICAL_POLARITY_CORRECTION_FAILED:{candidate_id}')
                score_reports.append(score_report)
            except subprocess.CalledProcessError as error:
                print(f'Refinement/evaluation failed for {candidate_id}: {error}')
else:
    print('No downloaded candidates. Run the provider cell with a Colab GPU.')
print(json.dumps({'refinementReports': [str(path) for path in refinement_reports], 'scoreReports': [str(path) for path in score_reports], 'unityInputAllowed': False}, indent=2, ensure_ascii=False))


In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
RANKING_MANIFEST = OUTPUT_ROOT / 'ranking-manifest.json'
if score_reports:
    rank_command = [
        sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'rank_candidates.py',
        '--output', RANKING_MANIFEST,
        '--contract', ROSTER_CONTRACT,
        '--character', CHARACTER_CODE,
    ]
    for score_report in score_reports:
        rank_command.extend(['--score-report', score_report])
    run(rank_command)
    ranking = json.loads(RANKING_MANIFEST.read_text(encoding='utf-8'))
    assert ranking['unityInputAllowed'] is False
    if ranking.get('selectedCandidate'):
        REVIEW_DIR.mkdir(parents=True, exist_ok=True)
        launcher = ['xvfb-run', '-a'] if shutil.which('xvfb-run') else []
        run(launcher + [
            'blender', '-b',
            '--python', TOOLS_DIR / 'scripts' / 'blender' / 'build_ai3d_review_asset.py',
            '--',
            '--ranking-manifest', RANKING_MANIFEST,
            '--socket-contract', TOOLS_DIR / 'contracts' / 'current_roster_socket_contract_v001.json',
            '--output-blend', REVIEW_DIR / f'{CHARACTER_CODE}_AI_AutoReview_NOT_PRODUCTION_v001.blend',
            '--report', REVIEW_DIR / 'ai3d-review-report.json',
        ])
    else:
        print('All candidates are below threshold: regeneration is required.')
else:
    print('Ranking skipped because no candidate score reports exist.')


In [ ]:
if not globals().get('GPU_WORK_ENABLED', False):
    raise RuntimeError('GPU_PROVIDER_WORKSTREAM_NOT_SELECTED')
archive_base = CONTENT_ROOT / f"re-camp-{CHARACTER_CODE}-ai3d-review-NOT-PRODUCTION"
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', OUTPUT_ROOT))
print('Archive:', archive_path)
if RUNTIME_NAME == 'colab':
    try:
        from google.colab import files
        files.download(str(archive_path))
    except Exception:
        print('Browser download is unavailable; copy the archive before the session ends.')
else:
    print(f'Archive retained at {archive_path}; download it from the Kaggle output panel.')
